# AxialMB v2: tier C pretraining on Colab

This notebook trains the 30k-step tier C model (d=192, 8 heads, 10 blocks) on an L4 or A100 runtime.
Every simulated world is generated on the fly; no benchmark data is used for training or selection.

1. On the laptop, run `powershell colab/make_bundle.ps1`. It writes `output/axialmb_v2/axialmb_v2_code.zip`.
2. Upload the zip to `MyDrive/axialmb_v2/`.
3. Pick an L4 or A100 runtime and run all cells. If the runtime disconnects, run all cells again: training resumes from `last.pt` on Drive.
4. Download `MyDrive/axialmb_v2/checkpoints/best.pt` into `output/axialmb_v2/checkpoints_C/` on the laptop.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/axialmb_v2'
!mkdir -p $DRIVE/checkpoints
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!rm -rf /content/axialmb_v2 && mkdir -p /content/axialmb_v2
!unzip -q -o $DRIVE/axialmb_v2_code.zip -d /content/axialmb_v2
%cd /content/axialmb_v2
!pip -q install pgmpy causal-learn tigramite scikit-learn scipy pandas

In [ ]:
# Short smoke run first, so a broken install fails in seconds rather than hours.
!python -u -m axialmb.train --tier smoke --out /content/smoke --device cuda

In [ ]:
# Tier C. Resumes from $DRIVE/checkpoints/last.pt when it exists.
# --max-minutes stops cleanly before a Colab session limit; rerun the cell to continue.
!python -u -m axialmb.train --tier C --workers 4 --max-minutes 680 --out $DRIVE/checkpoints --device cuda 2>&1 | tee -a $DRIVE/train_tierC.log

In [ ]:
# Synthetic gate: v2 against IAMB, PC, GES and PCMCI-PC1 on the frozen suite.
!python -u -m axialmb.validation --checkpoint $DRIVE/checkpoints/best.pt --out $DRIVE/validation_tierC.json --per-slice 8

In [ ]:
import json
print(json.dumps(json.load(open(f'{DRIVE}/checkpoints/train_summary.json')), indent=2))
res = json.load(open(f'{DRIVE}/validation_tierC.json'))
print(json.dumps(res['gate'], indent=2), 'all passed:', res['all_passed'])